In [ ]:
# 03_study2_ratings.ipynb
# Study 2 (corporate ratings): how much rating information do 2 indicators recover versus
# the full feature set, and is the information loss larger at the extremes than in the middle (H5)?
# Predictive recovery uses a nonparametric random forest (robust); statistical inference on the
# two proxies uses a low-dimensional ordered logit with agency fixed effects.
# All code in a single cell; grayscale figures saved as png + pdf at dpi 600.

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score, accuracy_score
from statsmodels.miscmodels.ordinal_model import OrderedModel
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

DATA = "../data"; FIG = "../results/figures"; TAB = "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="white", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.grid": False})
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight"); plt.close(fig)

# ---- load; collapse ultra-rare tail (CC,C,D) into CCC for a stable 7-grade scale ----
df = pd.read_csv(f"{DATA}/corporate_rating.csv"); df.columns = [c.strip().replace(" ", "_") for c in df.columns]
meta = ["Rating", "Name", "Symbol", "Rating_Agency_Name", "Date", "Sector"]
feats = [c for c in df.columns if c not in meta]
proxies = ["debtRatio", "returnOnAssets"]      # leverage, profitability
order = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]
df["R"] = df["Rating"].replace({"CC": "CCC", "C": "CCC", "D": "CCC"})
df["y"] = df["R"].map({r: i for i, r in enumerate(order)})
df["band"] = df["R"].apply(lambda r: "mid" if r in ("BBB", "BB", "B") else "extreme")
y = df["y"].values
X = df[feats].fillna(df[feats].median())
Xz = pd.DataFrame(StandardScaler().fit_transform(X), columns=feats, index=df.index).clip(-5, 5)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# ---- predictive recovery: full 25 features vs 2 proxies (random forest) ----
def rf_oof(cols):
    return cross_val_predict(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1), X[cols].values, y, cv=cv)
pred_full = rf_oof(feats); pred_two = rf_oof(proxies)
rec = pd.DataFrame([
    {"model": "Full 25 features", "QWK": round(cohen_kappa_score(y, pred_full, weights="quadratic"), 4),
     "accuracy": round(accuracy_score(y, pred_full), 4), "within1": round(float(np.mean(np.abs(y - pred_full) <= 1)), 4)},
    {"model": "2 proxies (debtRatio, ROA)", "QWK": round(cohen_kappa_score(y, pred_two, weights="quadratic"), 4),
     "accuracy": round(accuracy_score(y, pred_two), 4), "within1": round(float(np.mean(np.abs(y - pred_two) <= 1)), 4)},
])
rec.to_csv(f"{TAB}/study2_recovery.csv", index=False)
print("Rating recovery (RF, 5-fold):\n", rec.to_string(index=False))

# ---- band-wise information loss (H5) ----
band_rows = []
for b in ["mid", "extreme"]:
    m = df["band"].values == b
    wf = float(np.mean(np.abs(y[m] - pred_full[m]) <= 1)); wt = float(np.mean(np.abs(y[m] - pred_two[m]) <= 1))
    band_rows.append({"band": b, "n": int(m.sum()), "within1_full": round(wf, 4),
                      "within1_2proxy": round(wt, 4), "loss_pp": round(100 * (wf - wt), 1)})
band_loss = pd.DataFrame(band_rows); band_loss.to_csv(f"{TAB}/study2_band_loss.csv", index=False)
print("\nBand-wise information loss (H5):\n", band_loss.to_string(index=False))

# ---- inference: ordered logit on the 2 proxies + agency fixed effects ----
agency = pd.get_dummies(df["Rating_Agency_Name"], prefix="ag", drop_first=True).astype(float)
Xinf = pd.concat([Xz[proxies], agency], axis=1)
mo = OrderedModel(y, Xinf, distr="logit").fit(method="lbfgs", disp=0, maxiter=500)
inf = pd.DataFrame([{"variable": p, "coef": round(float(mo.params[p]), 3),
                     "z": round(float(mo.tvalues[p]), 2), "p": f"{mo.pvalues[p]:.2e}"} for p in proxies])
inf.to_csv(f"{TAB}/study2_inference.csv", index=False)
print("\nOrdered-logit inference (2 proxies + agency FE, pseudo-R2 = %.4f):\n" % mo.prsquared, inf.to_string(index=False))

# ---- migration matrix: actual grade vs 2-proxy predicted grade ----
mig = pd.crosstab(pd.Series([order[i] for i in y], name="actual"),
                  pd.Series([order[i] for i in pred_two], name="pred_2proxy")).reindex(index=order, columns=order).fillna(0).astype(int)
mig.to_csv(f"{TAB}/study2_migration_2proxy.csv")
print("\nMigration matrix (rows = actual, cols = predicted by 2 proxies):\n", mig.to_string())

# ---- Fig 7: recovery QWK full vs 2 proxies ----
fig, ax = plt.subplots(figsize=(3.6, 3.0))
ax.bar(["Full 25", "2 proxies"], rec["QWK"], color=["0.25", "0.7"], edgecolor="black", linewidth=0.7)
ax.set_ylabel("Quadratic weighted kappa"); ax.set_ylim(0, 0.75)
save_fig(fig, "fig07_study2_recovery_qwk")

# ---- Fig 8: band-wise within-1 recovery (full vs 2 proxies) ----
fig, ax = plt.subplots(figsize=(4.4, 3.0)); x = np.arange(2); w = 0.38
ax.bar(x - w/2, band_loss["within1_full"], w, color="0.3", edgecolor="black", linewidth=0.6, label="Full 25")
ax.bar(x + w/2, band_loss["within1_2proxy"], w, color="0.7", edgecolor="black", linewidth=0.6, label="2 proxies")
ax.set_xticks(x); ax.set_xticklabels(["Mid band", "Extremes"]); ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Within-1-grade recovery"); ax.legend(fontsize=8, frameon=False)
save_fig(fig, "fig08_study2_band_within1")

# ---- Fig 9: migration heatmap (grayscale) ----
fig, ax = plt.subplots(figsize=(4.6, 3.8))
sns.heatmap(mig, cmap="Greys", annot=True, fmt="d", cbar=True, linewidths=0.4, linecolor="0.8",
            annot_kws={"size": 7}, ax=ax)
ax.set_xlabel("Predicted grade (2 proxies)"); ax.set_ylabel("Actual grade")
save_fig(fig, "fig09_study2_migration_heatmap")

print("\n[03_study2] done. Figures + tables written to ../results/.")
